In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import json

carra_jsonl = "index_test.jsonl"
arome_jsonl = "index_test_AROME.jsonl"

def read_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_id_lookup(rows):
    return {row["id"]: row for row in rows}


carra_rows = read_jsonl(carra_jsonl)
arome_rows = read_jsonl(arome_jsonl)

carra_by_id = build_id_lookup(carra_rows)
arome_by_id = build_id_lookup(arome_rows)

sample_id = 440  

carra_row = carra_by_id[sample_id]
arome_row = arome_by_id[sample_id]

carra_path = carra_row["future_wind_path"]
arome_path = arome_row["future_wind_path"]


print("CARRA path:", carra_path)
print("AROME path:", arome_path)

def load_wind_npz(path):
    d = np.load(path, allow_pickle=True)
    u = d["u10_mean"]
    v = d["v10_mean"]
    wspd = d["wspd_mean"]
    attrs = d["attrs"].item()
    return u, v, wspd, attrs


u_c, v_c, s_c, attrs_c = load_wind_npz(carra_path)
u_a, v_a, s_a, attrs_a = load_wind_npz(arome_path)

print("CARRA attrs:", attrs_c)
print("AROME attrs:", attrs_a)

# Same color range
vmin = min(s_c.min(), s_a.min())
vmax = max(s_c.max(), s_a.max())

# Downsample vectors
step = max(1, u_c.shape[0] // 40)

y = np.arange(u_c.shape[0])
x = np.arange(u_c.shape[1])
X, Y = np.meshgrid(x, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- CARRA ----
im0 = axes[0].imshow(s_c, origin="lower", vmin=vmin, vmax=vmax)
axes[0].quiver(
    X[::step, ::step],
    Y[::step, ::step],
    u_c[::step, ::step],
    v_c[::step, ::step],
    color="white",
    scale=200
)
axes[0].set_title("CARRA wind")

# ---- AROME ----
im1 = axes[1].imshow(s_a, origin="lower", vmin=vmin, vmax=vmax)
axes[1].quiver(
    X[::step, ::step],
    Y[::step, ::step],
    u_a[::step, ::step],
    v_a[::step, ::step],
    color="white",
    scale=200
)
axes[1].set_title("AROME wind")

# Leave space for colorbar
fig.subplots_adjust(right=0.88)

# Colorbar axis
cbar_ax = fig.add_axes([0.9, 0.15, 0.02, 0.7])
fig.colorbar(im1, cax=cbar_ax, label="Wind speed (m/s)")

plt.show()
